In [1]:
from peft import PeftModelForCausalLM, PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
MODEL_ID= "Qwen/Qwen2.5-0.5B"


#tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [3]:
#ft_model = PeftModel.from_pretrained(base_model, "weights/lora-adapter")

In [4]:
#merged_model = ft_model.merge_and_unload()

In [5]:
#merged_model.save_pretrained("../_checkpoints/qwen2.5-0.5b/exp001/merged")
#tokenizer.save_pretrained("../_checkpoints/qwen2.5-0.5b/exp001/merged")

In [6]:
model = PeftModelForCausalLM.from_pretrained(base_model, "../_checkpoints/qwen2.5-0.5b/exp001")
tokenizer = AutoTokenizer.from_pretrained("../_checkpoints/qwen2.5-0.5b/exp001")

In [7]:
prompt = "User: Can you find recent research articles on CRISPR genome editing?\nAssistant: "
def ask_model(prompt, model=model, tokenizer=tokenizer):
    model.eval()
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )
    
    
    generated = model.generate(
        **inputs,
        max_new_tokens=256,
    )
    response = tokenizer.decode(
        generated[0],
        skip_special_tokens=True
    )
    
    #print(response)
    return response[len(prompt):]
    

In [8]:
ask_model("User: Can you find research papers on proteomics techniques\nAssistant: ")

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


'[{"tool": "search_pubmed", "arguments": {"gene_name": null, "sequence": null, "query": "proteomics techniques"}}]'

In [9]:
ask_model("User: Can you classify the following sequence to determine if it's DNA or RNA? Here's the sequence: AUGCUAGCUAGCUACG\nAssistant: ",)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


'[{"tool": "classify_sequence", "arguments": {"gene_name": null, "sequence": "AUGCUAGCUAGCUACG", "query": null}}]'

In [10]:
import json
entry="""[{"tool": "search_pubmed", "arguments": {"gene_name": null, "sequence": null, "query": "proteomics techniques"}}]"""
json.loads(entry)

[{'tool': 'search_pubmed',
  'arguments': {'gene_name': None,
   'sequence': None,
   'query': 'proteomics techniques'}}]

In [19]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="../datasets/v0.0.1/train.jsonl")

In [11]:
inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Find papers about proteomics."}],
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
)

In [12]:
inputs

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198,   9885,  15689,    911,   5425,
          23320,     13, 151645,    198, 151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1]])}

In [15]:
model = PeftModel.from_pretrained(base_model, "yusufteppei/bio-agent", revision="v0.0.2")
tokenizer = AutoTokenizer.from_pretrained("yusufteppei/bio-agent", revision="v0.0.2")

adapter_config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

/micromamba/envs/llm/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/8.68M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/697 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/2.43k [00:00<?, ?B/s]

In [16]:
ask_model("User: Can you classify the following sequence? Here's the sequence: AUGCUAGCUAGCUACG\nAssistant: ", model, tokenizer)

[transformers] Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


'[{"tool": "classify_sequence", "arguments": {"gene_name": null, "sequence": "AUGCUAGCUAGCUACG", "query": null}}]'